# Saadi on Colab's free GPU with an open model

The same quickstart without any API key: an 8B open model runs on the Colab GPU through [Ollama](https://ollama.com), and Saadi talks to it as an OpenAI-compatible server.

**Before running:** choose *Runtime → Change runtime type → T4 GPU*. The model download is about 5 GB, so the first cells take a few minutes.

**What to expect:** the paper's results used a much larger model. An 8B model follows the output contract most of the time, but it more often proposes an entity whose anchor text is not in the chunk, and such proposals are dropped by the grounding check. The index report counts them as `rejected_entities`. Expect a smaller entity memory than with a hosted model, and plan on roughly five to ten seconds per chunk.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The Ollama installer unpacks .tar.zst archives (zstd) and finds the GPU with lspci (pciutils);
# Colab's image has neither. The 'systemd is not running' warning is expected: the next cell starts the server.
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess
import time
import urllib.request

# An 8192-token context fits the indexing prompts; the server's output goes to ollama.log.
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192")
server = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see ollama.log")
print("Ollama is up")

In [ ]:
!ollama pull qwen3:8b

In [ ]:
# The current main branch; "saadi[openai]" from PyPI works once a release includes tolerant anchor matching.
%pip install -q 'saadi[openai] @ git+https://github.com/ZeinabTaghavi/SAADI@main'

In [ ]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ZeinabTaghavi/SAADI/main/examples/aurora_report/document.txt", "aurora_report.txt"
)

## Index once

`ollama:qwen3:8b` is the whole configuration. Qwen3 emits a thinking block before its answer; Saadi strips it.


In [ ]:
from saadi import Saadi

doc = Saadi(llm="ollama:qwen3:8b", state_dir="saadi_state/aurora", chunk_size=100)
report = doc.index("aurora_report.txt")
report

In [ ]:
for entity in doc.memory:
    print(entity.name, list(entity.aliases), "|", "; ".join(entity.descriptions)[:160])

In [ ]:
for item in doc.history.get(1, []):
    print(f"L1 fold over chunks {item.start_step}-{item.end_step}:", item.text, "\n")

## Retrieve and answer


In [ ]:
question = "Who keeps the Corvin station?"
for hit in doc.retrieve(question, k=3):
    print(hit.rank, f"chunk {hit.chunk_index}", round(hit.score, 3), hit.text[:110], "...")
print()
print(doc.answer(question).text)

## Notes

- `!ollama ps` shows `100% GPU` in the processor column when the model runs on the GPU. The server's log is in `ollama.log`.
- The server runs with an 8192-token context (`OLLAMA_CONTEXT_LENGTH` in the server cell). Raise it if answers are cut off.
- A larger model improves the entity memory more than any other setting; `ollama pull` any model that fits the GPU and change the spec.
- `saadi check-prompts default --llm ollama:qwen3:8b` reports in one call whether a model follows the output contract and how many proposed entities it keeps.